# **Pipeline de Preprocesamiento de Datos**
A continuación se mostrará el órden en el que se preprocesaron los datos en la Entrega 1

### **Importación de librerías**

In [87]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.calibration import LabelEncoder
import json
pd.options.display.max_columns = None

### **Importar datos**

In [88]:
data_path = 'data/'
df_lists = []

customers_df = pd.read_csv(data_path + 'olist_customers_dataset.csv')
geolocation_df = pd.read_csv(data_path + 'olist_geolocation_dataset.csv')
order_items_df = pd.read_csv(data_path + 'olist_order_items_dataset.csv')
order_payments_df = pd.read_csv(data_path + 'olist_order_payments_dataset.csv')
order_reviews_df = pd.read_csv(data_path + 'olist_order_reviews_dataset.csv')
orders_df = pd.read_csv(data_path + 'olist_orders_dataset.csv')
products_df = pd.read_csv(data_path + 'olist_products_dataset.csv')
sellers_df = pd.read_csv(data_path + 'olist_sellers_dataset.csv')
category_translation_df = pd.read_csv(data_path + 'product_category_name_translation.csv')

df_lists.append(customers_df)
df_lists.append(geolocation_df)
df_lists.append(order_items_df)
df_lists.append(order_payments_df)
df_lists.append(order_reviews_df)
df_lists.append(orders_df)
df_lists.append(products_df)
df_lists.append(sellers_df)
df_lists.append(category_translation_df)

df_names = ['customers', 'geolocation', 'order_items', 'order_payments', 'order_reviews', 'orders', 'products', 'sellers', 'category_translation']

### **Limpieza de datos**

In [89]:
from scipy import stats


def detect_outliers(df, column, method='zscore', threshold=3):
    
    if column not in df.columns or not np.issubdtype(df[column].dtype, np.number):
        return pd.Series(False, index=df.index)
    
    if method == 'zscore':
        z_scores = np.abs(stats.zscore(df[column].dropna()))
        outliers = pd.Series(False, index=df.index)
        outliers[df[column].dropna().index] = z_scores > threshold
        return outliers
    
    elif method == 'iqr':
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - threshold * IQR
        upper_bound = Q3 + threshold * IQR
        return (df[column] < lower_bound) | (df[column] > upper_bound)
    
    else:
        raise ValueError("Método no reconocido. Use 'zscore' o 'iqr'.")

def clean_dataframe(df, name):
    print(f"\n--- LIMPIEZA DE {name.upper()} ---")
    
    # Copia para no modificar el original
    df_clean = df.copy()
    changes = {}
    
    #Verificar duplicados
    n_duplicates = df_clean.duplicated().sum()
    if n_duplicates > 0:
        df_clean = df_clean.drop_duplicates()
        changes['duplicados_eliminados'] = n_duplicates
        print(f"- Se eliminaron {n_duplicates} filas duplicadas")
    
    # Convertir columnas de fechas a datetime
    date_columns = [col for col in df_clean.columns if any(date_term in col.lower() 
                                                         for date_term in ['date', 'time', '_at'])]
    for col in date_columns:
        if df_clean[col].dtype == object:
            try:
                df_clean[col] = pd.to_datetime(df_clean[col])
                changes[f'tipo_{col}'] = 'convertido a datetime'
                print(f"- Columna '{col}' convertida a datetime")
            except:
                pass
    
    # manejo de  valores extremos en columnas numéricas
    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        # Solo buscar outliers en columnas que no sean IDs o códigos
        if not any(id_term in col.lower() for id_term in ['id', 'code', 'zip', 'encoded']):
            outliers = detect_outliers(df_clean, col, method='iqr')
            n_outliers = outliers.sum()
            
            if n_outliers > 0 and n_outliers < len(df_clean) * 0.05:  # Si hay menos del 5% de outliers
                #recortar (clipping)
                Q1 = df_clean[col].quantile(0.01)  # Percentil 1%
                Q3 = df_clean[col].quantile(0.99)  # Percentil 99%
                
                orig_min = df_clean[col].min()
                orig_max = df_clean[col].max()
                
                # Aplicar recorte
                df_clean[col] = df_clean[col].clip(Q1, Q3)
                
                changes[f'outliers_{col}'] = f'{n_outliers} valores recortados (min: {orig_min:.2f} → {Q1:.2f}, max: {orig_max:.2f} → {Q3:.2f})'
                print(f"- Columna '{col}': {n_outliers} outliers recortados a percentiles 1-99%")
    

### **Manejo de Valores Nulos**

In [90]:
# Manejo de valores nulos en el dataset 'products_df'
products_df['product_category_name'] = products_df['product_category_name'].fillna('unknown')
products_df['product_name_lenght'] = products_df['product_name_lenght'].fillna(products_df['product_name_lenght'].median())
products_df['product_description_lenght'] = products_df['product_description_lenght'].fillna(products_df['product_description_lenght'].median())
products_df['product_photos_qty'] = products_df['product_photos_qty'].fillna(products_df['product_photos_qty'].median())
products_df['product_weight_g'] = products_df['product_weight_g'].fillna(products_df['product_weight_g'].median())
products_df['product_length_cm'] = products_df['product_length_cm'].fillna(products_df['product_length_cm'].median())
products_df['product_height_cm'] = products_df['product_height_cm'].fillna(products_df['product_height_cm'].median())
products_df['product_width_cm'] = products_df['product_width_cm'].fillna(products_df['product_width_cm'].median())

# Manejo de valores nulos para orders
orders_df['order_status_detailed'] = orders_df['order_status']
orders_df.loc[orders_df['order_approved_at'].isnull(), 'order_status_detailed'] = 'pending_approval'
orders_df['order_approved_at'] = orders_df['order_approved_at'].fillna(pd.NaT)
orders_df['order_delivered_carrier_date'] = orders_df['order_delivered_carrier_date'].fillna(pd.NaT)
orders_df['order_delivered_customer_date'] = orders_df['order_delivered_customer_date'].fillna(pd.NaT)

# Manejo de valores nulos para order reviews
order_reviews_df['review_comment_title'] = order_reviews_df['review_comment_title'].fillna('No Title')
order_reviews_df['review_comment_message'] = order_reviews_df['review_comment_message'].fillna('No Comment')

### **Codificación de variables categóricas**

In [91]:
def apply_label_encoding(df, column, return_mapping=False):
    if column not in df.columns or df[column].isnull().all():
        if return_mapping:
            return df, None
        return df

    le = LabelEncoder()
    df_copy = df.copy()

    temp_col = df_copy[column].fillna('MISSING')
    le.fit(temp_col)
    df_copy[column] = le.transform(temp_col)

    if return_mapping:
        mapping = dict(zip(le.classes_, range(len(le.classes_))))
        return df_copy, mapping

    return df_copy


# Función  para one-hot encoding
def apply_onehot_encoding(df, column, max_categories=15):
    if column not in df.columns:
        return df

    df_copy = df.copy()
    n_unique = df_copy[column].nunique()

    if n_unique <= max_categories:
        dummies = pd.get_dummies(df_copy[column], prefix=column, dummy_na=df_copy[column].isnull().any())
        df_copy = pd.concat([df_copy, dummies], axis=1)
        df_copy.drop(column, axis=1, inplace=True)
    
    return df_copy

encoding_mappings = {}

# aplicación de codificación a customers_df
customers_df, state_mapping = apply_label_encoding(customers_df, 'customer_state', return_mapping=True)
encoding_mappings['customer_state'] = state_mapping

# Codificación selectiva para city (si tiene muchas categorías)
if customers_df['customer_city'].nunique() <= 15:
    customers_df = apply_onehot_encoding(customers_df, 'customer_city')
else:
    customers_df, city_mapping = apply_label_encoding(customers_df, 'customer_city', return_mapping=True)
    encoding_mappings['customer_city'] = city_mapping

# Codificación para sellers_df
sellers_df, seller_state_mapping = apply_label_encoding(sellers_df, 'seller_state', return_mapping=True)
encoding_mappings['seller_state'] = seller_state_mapping

# Codificación para sellers_city (evaluar la cantidad de categorías)
if sellers_df['seller_city'].nunique() <= 15:
    sellers_df = apply_onehot_encoding(sellers_df, 'seller_city')
else:
    sellers_df = apply_label_encoding(sellers_df, 'seller_city')

# Codificación para orders_df - usar one-hot para order_status (pocas categorías)
orders_df = apply_onehot_encoding(orders_df, 'order_status')
orders_df = apply_onehot_encoding(orders_df, 'order_status_detailed')

# Codificación para order_payments_df
order_payments_df = apply_onehot_encoding(order_payments_df, 'payment_type')

# Codificación para categorías de productos
# Primero unir con la traducción
products_with_categories = pd.merge(products_df, category_translation_df, on='product_category_name', how='left')
products_with_categories['product_category_name_english'] = products_with_categories['product_category_name_english'].fillna('unknown')

# Aplicar label encoding a la categoría en inglés
products_with_categories, category_mapping = apply_label_encoding(
    products_with_categories, 'product_category_name_english', return_mapping=True)
encoding_mappings['product_category_name_english'] = category_mapping

def replace_ids(dataframes, id_columns, mapping_dicts, additional_columns=None):
    if additional_columns is None:
        additional_columns = []

    for df in dataframes:
        for column in id_columns.union(additional_columns):
            if column in df.columns:
                # Crear un mapeo único para la columna si no existe
                if column not in mapping_dicts:
                    unique_ids = df[column].unique()
                    mapping_dicts[column] = {old_id: new_id for new_id, old_id in enumerate(unique_ids, start=1)}
                
                # Reemplazar los IDs largos con los IDs simplificados
                df[column] = df[column].map(mapping_dicts[column])
    return dataframes

# Identificar columnas que terminan en '_id' en los DataFrames
id_columns = set()
for df in df_lists:
    id_columns.update([col for col in df.columns if col.endswith('_id')])

# Agregar columnas adicionales que quieras modificar
additional_columns = {'customer_zip_code_prefix', 'customer_city', 'customer_state'}

# Crear un diccionario para almacenar los mapeos de IDs
id_mappings = {}

# Aplicar la función a los DataFrames
df_lists = replace_ids(df_lists, id_columns, id_mappings, additional_columns)

# Guardar los mapeos de IDs en un archivo JSON

def ids(df, id_column, prefix, start_index=1):

    # Verificar si la columna existe
    if id_column not in df.columns:
        return df, {}
    
    # Obtener IDs únicos
    unique_ids = df[id_column].unique()
    
    # Crear mapeo
    id_mapping = {old_id: f"{prefix}{i}" for i, old_id in enumerate(unique_ids, start_index)}
    
    # Reemplazar directamente los IDs originales con los legibles
    df_copy = df.copy()
    df_copy[id_column] = df_copy[id_column].map(id_mapping)
    
    return df_copy, id_mapping


### **Eliminar Columnas Innecesarias**

In [92]:
# Eliminar columnas innecesarias
if 'customer_unique_id' in customers_df.columns:
    customers_df = customers_df.drop(columns=['customer_unique_id'])

# Generar IDs legibles para las entidades principales
customers_df, customer_id_mapping = ids(customers_df, 'customer_id', 'CUS_')

sellers_df, seller_id_mapping = ids(sellers_df, 'seller_id', 'SEL_')

products_with_categories, product_id_mapping = ids(products_with_categories, 'product_id', 'PRO_')

orders_df, order_id_mapping = ids(orders_df, 'order_id', 'ORD_')

# Actualiza customer_id en orders_df
if 'customer_id' in orders_df.columns:
    orders_df['customer_id'] = orders_df['customer_id'].map(lambda x: customer_id_mapping.get(x, x))

# Actualiza claves foráneas en order_items_df
if 'order_id' in order_items_df.columns:
    order_items_df['order_id'] = order_items_df['order_id'].map(lambda x: order_id_mapping.get(x, x))
if 'product_id' in order_items_df.columns:
    order_items_df['product_id'] = order_items_df['product_id'].map(lambda x: product_id_mapping.get(x, x))
if 'seller_id' in order_items_df.columns:
    order_items_df['seller_id'] = order_items_df['seller_id'].map(lambda x: seller_id_mapping.get(x, x))

# Actualiza claves foráneas en order_payments_df
if 'order_id' in order_payments_df.columns:
    order_payments_df['order_id'] = order_payments_df['order_id'].map(lambda x: order_id_mapping.get(x, x))

# Manejo de order_reviews_df
if 'order_id' in order_reviews_df.columns:
    order_reviews_df['order_id'] = order_reviews_df['order_id'].astype(str)
    mask_nan = order_reviews_df['order_id'].isin(['nan', 'None', '']) | order_reviews_df['order_id'].str.lower().isin(['nan', 'none'])
    order_reviews_df.loc[mask_nan, 'order_id'] = 'unknown'
    
    mask_numeric = ~mask_nan & order_reviews_df['order_id'].str.contains('.')
    if mask_numeric.any():
        safe_numeric_mask = mask_numeric & order_reviews_df['order_id'].str.replace('.', '', regex=False).str.isdigit()
        if safe_numeric_mask.any():
            numeric_ids = order_reviews_df.loc[safe_numeric_mask, 'order_id']
            order_reviews_df.loc[safe_numeric_mask, 'order_id'] = numeric_ids.astype(float).fillna(0).astype(int).astype(str)
    string_order_mapping = {str(k): v for k, v in order_id_mapping.items()}

    order_reviews_df['order_id'] = order_reviews_df['order_id'].map(
        lambda x: string_order_mapping.get(x, f"OR_{x}" if x != 'unknown' else "OR_unknown")
    )
    

if 'review_id' in order_reviews_df.columns:
    order_reviews_df, _ = ids(order_reviews_df, 'review_id', 'RV_')

with open("encoding_mappings.json", "w", encoding="utf-8") as f:
    json.dump(encoding_mappings, f, indent=4, ensure_ascii=False)

### **Guardar datos procesados**

In [93]:
#processed data
processed_path = data_path + 'processed/'
products_with_categories.to_csv(processed_path + 'processed_products.csv', index=False)
orders_df.to_csv(processed_path + 'processed_orders.csv', index=False)
customers_df.to_csv(processed_path + 'processed_customers.csv', index=False)
sellers_df.to_csv(processed_path + 'processed_sellers.csv', index=False)
order_payments_df.to_csv(processed_path + 'processed_payments.csv', index=False)
order_reviews_df.to_csv(processed_path + 'processed_reviews.csv', index=False)
order_items_df.to_csv(processed_path + 'processed_order_items.csv', index=False)

### **Creación del Dataset Final**

`num_orders`: total de pedidos del cliente.

`avg_order_value`: promedio de pago por pedido.

`std_order_value`: variabilidad en sus gastos.

`total_spent`: suma total pagada.

`frequent_category`: categoría de producto más comprada.

`avg_review_score`: satisfacción promedio.

`delivery_delay_avg`: promedio de días de atraso (si los hubo).

`first_purchase`, last_purchase`, `customer_lifetime_days`.

`region`: estado o ciudad del cliente.

In [94]:
data_path = 'data/processed/'

customers_df = pd.read_csv(data_path + 'processed_customers.csv')
order_items_df = pd.read_csv(data_path + 'processed_order_items.csv')
orders_df = pd.read_csv(data_path + 'processed_orders.csv')
payments_df = pd.read_csv(data_path + 'processed_payments.csv')
products_df = pd.read_csv(data_path + 'processed_products.csv')
reviews_df = pd.read_csv(data_path + 'processed_reviews.csv')
sellers_df = pd.read_csv(data_path + 'processed_sellers.csv')

reviews_df = reviews_df.drop_duplicates()

# Convertir todos los IDs a string
order_items_df["order_id"] = "ORD_" + order_items_df["order_id"].astype(str)
order_items_df["product_id"] = "PRO_" + order_items_df["product_id"].astype(str)
order_items_df["seller_id"] = "SEL_" + order_items_df["seller_id"].astype(str)

payments_df["order_id"] = payments_df["order_id"].astype(str)
reviews_df["order_id"] = reviews_df["order_id"].astype(str)

### **Total gastado (total_spent)**

In [95]:
# Merge payments y orders
payments_orders = pd.merge(payments_df, orders_df[['order_id', 'customer_id']], on='order_id', how='inner')

# Agrupar
df_total_spent = payments_orders.groupby('customer_id')['payment_value'].sum().reset_index()
df_total_spent.rename(columns={'payment_value': 'total_spent'}, inplace=True)

### **Número de pedidos (num_orders)**

In [96]:
df_num_orders = orders_df.groupby('customer_id').size().reset_index(name='num_orders')

### **Eliminar Reviews sin IDS**

In [97]:
reviews_df.sort_values(by='order_id', ascending=True)

unknown_orders = reviews_df[reviews_df['order_id'].str.contains('unknown')]

### **Promedio de gasto por pedido (avg_order_value)**

In [98]:
# Agrupar por orden
order_avg_value = payments_orders.groupby('order_id')['payment_value'].sum().reset_index()

# Conectar con cliente
order_avg_value = pd.merge(order_avg_value, orders_df[['order_id', 'customer_id']], on='order_id', how='inner')

# Agrupar por cliente
df_avg_order_value = order_avg_value.groupby('customer_id')['payment_value'].mean().reset_index()
df_avg_order_value.rename(columns={'payment_value': 'avg_order_value'}, inplace=True)

### **Promedio de review (avg_review_score)**

In [99]:
# Unir reviews y orders

# Cambiar id OR por ORD
reviews_df['order_id'] = reviews_df['order_id'].str.replace('OR_', 'ORD_', regex=False)
reviews_orders = pd.merge(reviews_df[['order_id', 'review_score']], orders_df[['order_id', 'customer_id']], on='order_id', how='inner')

df_avg_review = reviews_orders.groupby('customer_id')['review_score'].mean().reset_index()
df_avg_review.rename(columns={'review_score': 'avg_review_score'}, inplace=True)

### **Customer Lifetime Days (customer_lifetime_days)**

In [100]:
# Asegurarse que las fechas estén en datetime
orders_df['order_purchase_timestamp'] = pd.to_datetime(orders_df['order_purchase_timestamp'])

# Agrupar
orders_dates = orders_df.groupby('customer_id').agg(
    first_purchase=('order_purchase_timestamp', 'min'),
    last_purchase=('order_purchase_timestamp', 'max')
).reset_index()

orders_dates['customer_lifetime_days'] = (orders_dates['last_purchase'] - orders_dates['first_purchase']).dt.days

### **Región (customer_state)**

In [101]:
df_region = customers_df[['customer_id', 'customer_state']]

### **Categoría más frecuente (frequent_category)**

In [102]:
# Conectar productos a pedidos
orders_products = pd.merge(order_items_df[['order_id', 'product_id']], orders_df[['order_id', 'customer_id']], on='order_id', how='inner')
orders_products = pd.merge(orders_products, products_df[['product_id', 'product_category_name']], on='product_id', how='left')

# Agrupar y encontrar la categoría más frecuente
frequent_category = orders_products.groupby(['customer_id', 'product_category_name']).size().reset_index(name='counts')

# Traducir el nombre de la categoría
translations = pd.read_csv('data/product_category_name_translation.csv')
with open('encoding_mappings.json', 'r') as f:
    encoding_mappings = json.load(f)
# Se hace merge para traducir los nombres
frequent_category = frequent_category.merge(
    translations, 
    on='product_category_name', 
    how='left'
)
# Codificación de la categoría
encoding_dict = encoding_mappings["product_category_name_english"]
frequent_category = frequent_category.drop(columns=['product_category_name'])
frequent_category['product_category_name'] = frequent_category['product_category_name_english'].map(encoding_dict)
# Eliminar columnas innecesarias
frequent_category = frequent_category.drop(columns=['product_category_name_english'])
print(frequent_category.head())

idx = frequent_category.groupby('customer_id')['counts'].idxmax()
df_frequent_category = frequent_category.loc[idx, ['customer_id', 'product_category_name']]
df_frequent_category.rename(columns={'product_category_name': 'frequent_category'}, inplace=True)

  customer_id  counts  product_category_name
0       CUS_1       1                   34.0
1      CUS_10       1                   43.0
2     CUS_100       1                   40.0
3    CUS_1000       1                   37.0
4   CUS_10000       1                   43.0


### **Atraso promedio en la entrega (delivery_delay_avg)**

In [103]:
# Convertir fechas
orders_df['order_delivered_customer_date'] = pd.to_datetime(orders_df['order_delivered_customer_date'])
orders_df['order_estimated_delivery_date'] = pd.to_datetime(orders_df['order_estimated_delivery_date'])

# Calcular delay
orders_df['delivery_delay'] = (orders_df['order_delivered_customer_date'] - orders_df['order_estimated_delivery_date']).dt.days

# Filtrar pedidos entregados
orders_delivered = orders_df[orders_df['order_delivered_customer_date'].notna()]

# Agrupar por cliente
df_delivery_delay = orders_delivered.groupby('customer_id')['delivery_delay'].mean().reset_index()
df_delivery_delay.rename(columns={'delivery_delay': 'delivery_delay_avg'}, inplace=True)


### **Unir en un solo DataFrame**

In [104]:
# Empezar con el gasto total
df_customer_value = df_total_spent

# Luego unir todos los demás
dfs = [
    df_num_orders,
    df_avg_order_value,
    df_avg_review,
    orders_dates[['customer_id', 'customer_lifetime_days']],
    df_region,
    df_frequent_category,
    df_delivery_delay
]

for df in dfs:
    df_customer_value = pd.merge(df_customer_value, df, on='customer_id', how='left')

# Mostrar resultado
df_customer_value.head()


,customer_id,total_spent,num_orders,avg_order_value,avg_review_score,customer_lifetime_days,customer_state,frequent_category,delivery_delay_avg
0,CUS_1,146.87,1,146.87,5.0,0,25,34.0,-11.0
1,CUS_10,122.47,1,122.47,5.0,0,10,43.0,-23.0
2,CUS_100,76.15,1,76.15,5.0,0,9,40.0,2.0
3,CUS_1000,174.43,1,174.43,5.0,0,15,37.0,-5.0
4,CUS_10000,511.43,1,511.43,5.0,0,10,43.0,-13.0
